# finish_reason: tell a finished answer from a cut-off one

**Scenario:** halfway through a race, a Formula 1 strategy service starts returning pit plans with
no stops at all. Two unrelated faults cause that one symptom, and they need opposite fixes.

It is like the reason a phone call ended: a hang up, a dead battery and a lost signal each need a
different response.

### What you will learn

- Read `finish_reason` on every response before you read its content.
- Recognise truncation, where the model hit your output cap mid answer.
- Retry only what a bigger budget can fix, up to a ceiling you chose.

## What finish_reason tells you about every response

Every response carries `finish_reason`, the field that says why the model stopped writing.

| Value | Meaning | What your code should do |
|---|---|---|
| `stop` | The model finished on its own | Use the answer |
| `tool_calls` | It wants a function run | Execute, append the result, loop |
| `length` | **It was cut off** at your output cap | Do not use the answer. Raise, or ask for more |
| `content_filter` | Blocked by a safety filter | Do not retry the same thing |

The trap is that `length` still returns text, so a cut-off answer looks complete.

### Step 1: Parse whatever the strategy service returns

![Parse whatever the strategy service returns](images/finish-reason-step-1.svg)

The first version plans no stops whenever parsing fails, so it never crashes and nobody notices.

## What a silent empty plan costs

An empty plan is worse than an error, because the pit wall trusts it.

```
cost = races where the car did not pit x the value of a race
```

## Both budgets return a plan with no stops

The strategy call below runs once with a generous budget and once with a tight one.

In [1]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("01-stateful-agent-runtime/02-finish-reason-as-a-state-machine")

SYSTEM = ('You are a Formula 1 race strategist. Reply with JSON only: '
          '{"stops":[{"lap":int,"tyre":"soft|medium|hard"}],"note":str}')
RACE = ("58 lap race, high tyre wear, safety car likely around lap 30. "
        "Give the full pit strategy with reasoning in the note.")


def ask(max_tokens):
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=max_tokens,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": RACE}])
    return reply.choices[0]

Most services first write a parser that returns an empty plan on any error.

In [2]:
def stops_naive(choice):
    """Parse the plan. Falls back to no stops if anything goes wrong."""
    try:
        return json.loads(choice.message.content)["stops"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return []


generous = ask(400)
tight = ask(40)

print(f"generous budget: {len(stops_naive(generous))} stops planned")
print(f"tight budget   : {len(stops_naive(tight))} stops planned")
print("\nthe car never pits, and nothing raised")

generous budget: 0 stops planned
tight budget   : 0 stops planned

the car never pits, and nothing raised


Both plans are empty, so the next cell prints why each call stopped.

In [3]:
for label, choice in (("generous", generous), ("tight", tight)):
    text = choice.message.content or ""
    print(f"{label:9} finish_reason={choice.finish_reason!r}")
    print(f"{'':9} tail: ...{text[-58:]!r}\n")

assert stops_naive(generous), "a complete answer produced an empty plan"

generous  finish_reason='stop'
          tail: ...'taking or defending on fresh tyres towards the end."\n}\n```'

tight     finish_reason='length'
          tail: ...'    "lap": 28,\n      "tyre": "medium"\n    },\n    {\n      "'



AssertionError: a complete answer produced an empty plan

### Step 2: Both calls plan zero pit stops

![Both calls plan zero pit stops](images/finish-reason-step-2.svg)

Neither call raises an error, so the pit wall gets an empty plan it has no reason to doubt.

## One symptom with two different causes

- **The generous call** stopped with `stop`: the plan is complete, just wrapped in a code fence.
- **The tight call** stopped with `length`: the model was cut off halfway through the plan.

A `try/except` block cannot tell them apart, yet half a strategy is worse than none.

### Step 3: The stop reason was on the response all along

![The stop reason was on the response all along](images/finish-reason-step-3.svg)

The first parser never read the one field that separates the two cases.

## Branch on finish_reason before touching the content

The fix reads `finish_reason` first, so each way of stopping gets its own handling.

### Step 4: Give each stop reason its own handling

![Give each stop reason its own handling](images/finish-reason-step-4.svg)

One branch per state replaces the catch-all, and the two dangerous states raise errors.

In [4]:
class Truncated(Exception):
    """Generation hit the token cap. The answer is incomplete, not wrong."""

A named exception tells a cut-off answer apart from a malformed one.

In [5]:
def parse_strategy(choice):
    """Read finish_reason before content. Raise rather than guess."""
    if choice.finish_reason == "length":
        raise Truncated("cut off at the token cap, ask for more room")
    if choice.finish_reason == "content_filter":
        raise ValueError("blocked by a filter, retrying the same prompt will not help")

    text = (choice.message.content or "").strip()
    if text.startswith("```"):
        text = text.split("```")[1].removeprefix("json").strip()
    return json.loads(text)["stops"]

Both saved responses now separate, and the dangerous one fails clearly.

In [6]:
for label, choice in (("generous", generous), ("tight", tight)):
    try:
        stops = parse_strategy(choice)
        print(f"{label:9} {len(stops)} stops: {[s['lap'] for s in stops]}")
    except Truncated as exc:
        print(f"{label:9} REFUSED, {exc}")

print("\nbefore: both silently returned 0 stops")
print("after : one plan parsed, one refused loudly")

generous  2 stops: [28, 55]
tight     REFUSED, cut off at the token cap, ask for more room

before: both silently returned 0 stops
after : one plan parsed, one refused loudly


Both rows were printed by the cell above.

| Budget | Before the fix | After the fix |
|---|---|---|
| Generous | 0 stops, no error | 2 stops, on laps 28 and 55 |
| Tight | 0 stops, no error | refused, cut off at the output cap |

### Step 5: Retry only when the answer was cut off

![Retry only when the answer was cut off](images/finish-reason-step-5.svg)

A bigger budget fixes `length` and nothing else, so a retry never doubles the bill for nothing.

In [7]:
def ask_until_complete(max_tokens, ceiling=1200):
    """Grow the budget until the model finishes, or give up honestly."""
    while max_tokens <= ceiling:
        choice = ask(max_tokens)
        if choice.finish_reason != "length":
            return choice, max_tokens
        max_tokens *= 4
    raise Truncated(f"still truncated at {ceiling} tokens, the request is too large")

A budget far too small shows the loop growing it until the plan is finished.

In [8]:
choice, used = ask_until_complete(40)
stops = parse_strategy(choice)

print(f"started at 40 tokens, succeeded at {used}")
print(f"finish_reason: {choice.finish_reason}")
print(f"plan: {[(s['lap'], s['tyre']) for s in stops]}")

started at 40 tokens, succeeded at 640
finish_reason: stop
plan: [(28, 'medium'), (48, 'hard')]


## A test that fails if truncation reaches the parser

This test hands the parser a cut-off answer, guarding against a bare `try/except` coming back.

### Step 6: Test that a cut-off plan is refused

![Test that a cut-off plan is refused](images/finish-reason-step-6.svg)

Delete the `length` branch and this test fails on the next commit.

In [9]:
class FakeChoice:
    """A response shaped object, so the test needs no API call."""
    def __init__(self, reason, text):
        self.finish_reason = reason
        self.message = type("M", (), {"content": text})()

The fake response needs no network, so the test runs in milliseconds.

In [10]:
def test_truncated_output_is_never_parsed():
    cut_off = FakeChoice("length", '{"stops": [{"lap": 15, "tyre": "med')
    try:
        parse_strategy(cut_off)
    except Truncated:
        return
    raise AssertionError("a truncated answer was parsed instead of refused")


test_truncated_output_is_never_parsed()
print("gate holds: length never reaches the parser")

gate holds: length never reaches the parser


### Enterprise exploration

- The retry loop quadruples the budget each time, so what caps that cost during an incident?
- When a plan is refused, what does the pit wall see, and is a stale plan better than none?
- Filtered answers are never retried, so who finds out when it happens?

### Key terms and traps

- **finish_reason**: the field on every response that says why the model stopped.
- **Truncation**: the model hit your output cap mid answer, which `length` reports.
- **Trap**: a `try/except` that returns an empty result hides a failure as a wrong answer.